### Firefox DOM diagnostic for IJSEM pages.

### Usage:
###    python firefoxwebdriver_test_fulltext.py <URL>

### If no URL is provided, a sample IJSEM DOI URL is used.

In [1]:
DEFAULT_URL = "https://www.microbiologyresearch.org/content/journal/ijsem/10.1099/ijsem.0.007124"

In [2]:
import sys
import time
from typing import List

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options

In [12]:
def make_driver(headless: bool = True) -> webdriver.Firefox:
    options = Options()
    if headless:
        options.add_argument("-headless")
    driver = webdriver.Firefox(options=options)
    print("Firefox browser version:", driver.capabilities.get("browserVersion"))
    print(
        "Firefox driver info:",
        driver.capabilities.get("moz:geckodriverVersion", driver.capabilities.get("moz:firefoxOptions", {})),
    )
    return driver


Firefox browser version: 140.10.0
Firefox driver info: 0.36.0
WebDriverException: Message: Expected "url" to be a valid URL, got -f
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:199:5
InvalidArgumentError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:401:5
GeckoDriver.prototype.navigateTo@chrome://remote/content/marionette/driver.sys.mjs:1080:11



SystemExit: 1

In [25]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
import time

url = "https://www.microbiologyresearch.org/content/journal/ijsem/10.1099/ijsem.0.007124"

opts = Options()
# keep headless off for this test
# opts.add_argument("-headless")

service = Service(log_output="geckodriver.log")
driver = webdriver.Firefox(options=opts, service=service)

try:
    print("started:", driver.capabilities.get("browserVersion"))
    driver.get(url)
    time.sleep(3)
    print("got page")
    html = driver.execute_script("return document.documentElement.outerHTML")
    #print("html length:", len(html))
    #print(html[:500])
    #print("\n--- Searching for 'Description of' ---")
    #for line in html.splitlines():
        #if "Description of" in line:
            #print(line)
    with open("page.html", "w") as f:
        f.write(html)
finally:
    driver.quit()

started: 140.10.0
got page


In [26]:
#!/usr/bin/env python3
"""
Firefox DOM diagnostic for IJSEM pages.

Usage:
    python firefoxwebdriver_test_fulltext.py <URL>

If no URL is provided, a sample IJSEM DOI URL is used.
"""

import sys
import time
from typing import List

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.firefox.options import Options


DEFAULT_URL = "https://www.microbiologyresearch.org/content/journal/ijsem/10.1099/ijsem.0.007124"


def make_driver(headless: bool = True) -> webdriver.Firefox:
    options = Options()
    if headless:
        options.add_argument("-headless")
    driver = webdriver.Firefox(options=options)
    print("Firefox browser version:", driver.capabilities.get("browserVersion"))
    print(
        "Firefox driver info:",
        driver.capabilities.get("moz:geckodriverVersion", driver.capabilities.get("moz:firefoxOptions", {})),
    )
    return driver


def print_all_elements(label: str, elements: List) -> None:
    print(f"{label} count:", len(elements))
    for i, el in enumerate(elements, start=1):
        txt = (el.text or "").strip()
        print(f"  [{i}] {txt}")


def print_all_soup_matches(label: str, elements: List) -> None:
    print(f"{label} count:", len(elements))
    for i, el in enumerate(elements, start=1):
        txt = el.get_text(" ", strip=True)
        print(f"  [{i}] {txt}")


def print_description_blocks_from_live_dom(driver, soup, filtered_url):
    counter = 1
    combined_description = []
    title_blocks = driver.find_elements(By.CSS_SELECTOR, "div.tl-main-part.title")
    if not title_blocks:
        title_blocks = soup.select("div.tl-main-part.title")

    print("\nDescription blocks:")
    print("  title_blocks count:", len(title_blocks))
    for element in title_blocks:  # finds section headers
        counter += 1
        description = element.text
        print("text", description)
        if "Description of" in description:
            strains = []
            snumber = "s" + str(counter - 4)
            print("snumber is", snumber)

            live_elements = driver.find_elements(By.ID, snumber)
            if not live_elements:
                live_elements = [soup.find(id=snumber)] if soup.find(id=snumber) else []

            for element in live_elements:
                if element is None:
                    continue
                description = element.text if hasattr(element, "text") else element.get_text(" ", strip=True)
                cleaned_text = remove_non_ascii(description)
                combined_description.append(cleaned_text)
                print(cleaned_text)
                print("description of", description)

    return combined_description


def main() -> int:
    url = sys.argv[1] if len(sys.argv) > 1 else DEFAULT_URL

    driver = make_driver(headless=True)
    try:
        driver.get(url)

        print("\nURL:", url)
        print("  current_url (initial):", driver.current_url)

        wait = WebDriverWait(driver, 30)
        wait.until(lambda d: "citation_title" in d.execute_script("return document.documentElement.outerHTML"))
        wait.until(lambda d: "item-meta-data__item-title" in d.execute_script("return document.documentElement.outerHTML"))
        time.sleep(2)

        print("  current_url (after wait):", driver.current_url)
        print("  iframe count:", len(driver.find_elements(By.TAG_NAME, "iframe")))

        # Use live DOM HTML rather than page_source.
        html = driver.execute_script("return document.documentElement.outerHTML")
        print("  html_length:", len(html or ""))
        print("  contains 'Description of':", "Description of" in html)
        print("  contains 'item-meta-data__item-title':", "item-meta-data__item-title" in html)
        print("  contains 'tl-main-part':", "tl-main-part" in html)
        print("  contains 'tl-lowest-section':", "tl-lowest-section" in html)

        with open("page.html", "w", encoding="utf-8") as f:
            f.write(html)
        print("  wrote full DOM to page.html")

        print("\n--- Full DOM (line by line) ---")
        for i, line in enumerate(html.splitlines(), 1):
            print(f"{i:05d}: {line}")

        soup = BeautifulSoup(html, "html.parser")

        # Browser-level selector checks
        print_all_elements("item-meta-data__item-title", driver.find_elements(By.CLASS_NAME, "item-meta-data__item-title"))
        print_all_elements("doi links", driver.find_elements(By.PARTIAL_LINK_TEXT, "doi.org"))
        print_all_elements("author spans", driver.find_elements(By.XPATH, "//*[@id='bellowheadercontainer']/main/div[2]/div/ul/li[1]/span"))
        print_all_elements("date spans", driver.find_elements(By.XPATH, "//*[@id='bellowheadercontainer']/main/div[2]/div/ul/li[3]/span/span[2]"))
        print_all_elements("tl-main-part.title", driver.find_elements(By.CSS_SELECTOR, "div.tl-main-part.title"))
        print_all_elements("tl-lowest-section", driver.find_elements(By.CLASS_NAME, "tl-lowest-section"))

        # Live-soup selector checks
        print("\nLive soup selector checks:")
        print_all_soup_matches("soup.select('div.tl-main-part.title')", soup.select("div.tl-main-part.title"))
        print_all_soup_matches("soup.select('div.tl-lowest-section')", soup.select("div.tl-lowest-section"))

        # Meta tags your scraper might use
        print("\nCitation meta tags:")
        for meta_name in ["citation_title", "citation_doi", "citation_author", "citation_publication_date", "citation_date"]:
            vals = [m.get("content", "").strip() for m in soup.find_all("meta", attrs={"name": meta_name}) if m.get("content")]
            print(f"  {meta_name} count:", len(vals))
            for i, v in enumerate(vals, start=1):
                print(f"    [{i}] {v}")

        # Print the description blocks the way the scraper expects them.
        combined_description = print_description_blocks_from_live_dom(driver, soup, url)
        print("\nCombined description count:", len(combined_description))
        for i, desc in enumerate(combined_description, start=1):
            print(f"  [combined {i}] {desc}")

        return 0
    except WebDriverException as e:
        print("WebDriverException:", e)
        return 1
    finally:
        try:
            driver.quit()
        except Exception:
            pass


if __name__ == "__main__":
    raise SystemExit(main())


Firefox browser version: 140.10.0
Firefox driver info: 0.36.0
WebDriverException: Message: Expected "url" to be a valid URL, got -f
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:199:5
InvalidArgumentError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:401:5
GeckoDriver.prototype.navigateTo@chrome://remote/content/marionette/driver.sys.mjs:1080:11



SystemExit: 1

In [27]:
%tb


SystemExit: 1